# Tile index: check the remote agent's metadata reader

`read_tile_index(DATA_DIR)` (written by the remote agent, task T001) reads the metadata CSV and
pairs each SEM image with its `.oas` in `Contour/`. It returns one record per tile.

This notebook shows what it returns, then looks at one tile:
1. the tile table;
2. one SEM image, to see whether it has a **data bar** (a strip with text/scale bar at the bottom);
3. the design and SEM contacts of that tile in **mask coordinates** (tile-local + tile centre);
4. design ↔ SEM **matching**: only contacts found in both are kept, with their offset.

Set `DATA_DIR` below and run all cells.

In [ ]:
DATA_DIR = r"path\to\data"   # folder with the .jpg files, the metadata .csv and Contour/
TILE = 0                      # which tile to look at (index in the table)
LAYER = (1, 0)                # (layer, datatype) of the contacts in the .oas
TONE_REVERSED = False         # True if this tile's .oas draws the area AROUND the holes
DATABAR_ROWS = 0              # rows of info bar at the bottom of the image (0: none)
MAX_MATCH_NM = 50.0           # pair design/SEM contacts closer than this; keep < half the pitch

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.design import load_layout, read_contacts, read_contacts_tone_reversed
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.io.sem_image import crop_databar, load_sem_image
from affine_ransac.matching import match_points

## 1. Tile table

In [ ]:
tiles = read_tile_index(DATA_DIR)

table = pd.DataFrame([dataclasses.asdict(t) for t in tiles]).drop(columns="extra")
table["image_path"] = table.image_path.map(lambda p: p.name)
table["oas_path"] = table.oas_path.map(lambda p: p.name if p else None)

print(f"{len(tiles)} tiles, {table.oas_path.isna().sum()} without an .oas file")
table

## 2. One SEM image: is there a data bar?

Look at the bottom of the image. If there is a strip with text or a scale bar, note the
row (y, in px) where it starts, and set `DATABAR_ROWS` = image height − that row.

In [ ]:
tile = tiles[TILE]
raw = load_sem_image(tile.image_path)
print(f"Tile {tile.tile_id}: image file is {raw.shape[1]} x {raw.shape[0]} px; "
      f"metadata says {tile.image_width_px} x {tile.image_height_px} px")

plt.figure(figsize=(8, 8))
plt.imshow(raw, cmap="gray")
plt.title(f"{tile.image_path.name}  (x right, y down, px)")
plt.show()

## 3. This tile's contacts in mask coordinates

Design: `.oas` positions (centred on (0, 0)) + tile centre.
SEM: detected centres → tile-local nm → + tile centre.
The two sets of points should lie on top of each other.

In [ ]:
layout = load_layout(tile.oas_path)
if TONE_REVERSED:
    design_local, _ = read_contacts_tone_reversed(layout, *LAYER, frame_nm=(tile.fov_x_nm, tile.fov_y_nm))
else:
    design_local, _ = read_contacts(layout, *LAYER)

image = crop_databar(raw, DATABAR_ROWS)
found = detect_contacts(image)
sem_local = pixel_to_tile_nm(found.centers, image.shape, tile.fov_x_nm / image.shape[1])

center = (tile.center_x_nm, tile.center_y_nm)
design_mask = design_local + center
sem_mask = sem_local + center

print(f"Tile centre (mask): x = {center[0]:.1f} nm, y = {center[1]:.1f} nm")
print(f"Design contacts: {len(design_mask)}, SEM contacts: {len(sem_mask)}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(design_mask[:, 0], design_mask[:, 1], marker="s", facecolors="none", edgecolors="tab:blue", label="design (.oas)")
ax.scatter(sem_mask[:, 0], sem_mask[:, 1], marker="+", color="tab:red", label="SEM")
ax.set_aspect("equal")
ax.set_xlabel("mask x (nm)")
ax.set_ylabel("mask y (nm)")
ax.legend()
plt.show()

## 4. Match design ↔ SEM

Each design contact is paired with its SEM contact (mutual nearest, closer than
`MAX_MATCH_NM`). Contacts found in only one of the two are left out and circled in the plot,
e.g. contacts cut off by the image edge, which SEM detection drops on purpose.
`dx_nm, dy_nm` = SEM − design, before any affine correction.

In [ ]:
i_design, i_sem = match_points(design_mask, sem_mask, MAX_MATCH_NM)
pairs = pd.DataFrame({
    "x_nm": design_mask[i_design, 0], "y_nm": design_mask[i_design, 1],
    "dx_nm": sem_mask[i_sem, 0] - design_mask[i_design, 0],
    "dy_nm": sem_mask[i_sem, 1] - design_mask[i_design, 1],
})
unmatched_design = design_mask[np.setdiff1d(np.arange(len(design_mask)), i_design)]
unmatched_sem = sem_mask[np.setdiff1d(np.arange(len(sem_mask)), i_sem)]

print(f"Matched pairs: {len(pairs)}")
print(f"Design only: {len(unmatched_design)}, SEM only: {len(unmatched_sem)}")
print(f"Mean offset: dx = {pairs.dx_nm.mean():.2f} nm, dy = {pairs.dy_nm.mean():.2f} nm")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pairs.x_nm, pairs.y_nm, marker=".", color="gray", label="matched")
ax.scatter(unmatched_design[:, 0], unmatched_design[:, 1], s=120, facecolors="none", edgecolors="tab:blue", label="design only")
ax.scatter(unmatched_sem[:, 0], unmatched_sem[:, 1], s=120, facecolors="none", edgecolors="tab:red", label="SEM only")
ax.set_aspect("equal")
ax.set_xlabel("mask x (nm)")
ax.set_ylabel("mask y (nm)")
ax.legend()
plt.show()
pairs